# FFT Causal Conv1D Forward

This notebook exercises the cuDNN Frontend `fft_causal_conv1d(x, weight)` wrapper through both its medium and long FFT paths.

## Prerequisites

An NVIDIA GPU, cuDNN 9.26.0 or newer, and a cuDNN Frontend Python binding built against cuDNN 9.26.0 or newer are required.

In [1]:
import math

import cudnn
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "This sample requires a CUDA device"
assert cudnn.backend_version() >= 92600, "FFT causal conv1d requires cuDNN 9.26.0 or newer"
print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92600


## Reference

FFT causal conv1d uses FIR-order weights: `weight[0]` multiplies the current sample. PyTorch `conv1d` is cross-correlation, so the reference reverses the filter before applying left-only padding.

In [2]:
def fft_causal_conv1d_reference(x, weight):
    dim, kernel_size = weight.shape
    x_padded = F.pad(x, (kernel_size - 1, 0))
    return F.conv1d(x_padded, weight.flip(-1).unsqueeze(1), groups=dim)

## Medium and long path checks

The medium case intentionally uses non-power-of-two `seq_len` and `kernel_size`, which validates the wrapper's padding and trimming. The FP64 case uses `kernel_size=8192`, above FP64's medium limit, to force the public wrapper through the long FFT path.

In [3]:
torch.manual_seed(42)
configs = [
    {"name": "medium-padded", "batch": 2, "dim": 4, "seq_len": 750, "kernel_size": 192, "dtype": torch.float32, "atol": 2e-6, "rtol": 2e-6},
    {"name": "long", "batch": 1, "dim": 1, "seq_len": 8192, "kernel_size": 8192, "dtype": torch.float64, "atol": 5e-11, "rtol": 5e-11},
]

for config in configs:
    x = 0.1 * torch.randn(config["batch"], config["dim"], config["seq_len"], device="cuda", dtype=config["dtype"])
    weight = torch.randn(config["dim"], config["kernel_size"], device="cuda", dtype=config["dtype"]) / math.sqrt(config["kernel_size"])

    actual = cudnn.ops.fft_causal_conv1d(x, weight)
    expected = fft_causal_conv1d_reference(x.double(), weight.double()).to(config["dtype"])
    max_abs = (actual - expected).abs().max().item()
    print(f'{config["name"]}: shape={tuple(actual.shape)}, max_abs={max_abs:.6e}')
    torch.testing.assert_close(actual, expected, atol=config["atol"], rtol=config["rtol"])

medium-padded: shape=(2, 4, 750), max_abs=1.043081e-07


long: shape=(1, 1, 8192), max_abs=1.776357e-15


## `torch.compile`

The registered custom operators preserve the same result under `torch.compile`.

In [4]:
@torch.compile
def compiled_fft_causal_conv1d(x, weight):
    return cudnn.ops.fft_causal_conv1d(x, weight)


x = torch.randn(1, 2, 512, device="cuda")
weight = torch.randn(2, 128, device="cuda")
eager = cudnn.ops.fft_causal_conv1d(x, weight)
compiled = compiled_fft_causal_conv1d(x, weight)
torch.testing.assert_close(compiled, eager, atol=0.0, rtol=0.0)
print("torch.compile output matches eager output")

torch.compile output matches eager output
